In [1]:
# Paths and constants
TRAIN_PATH = "/kaggle/input/learning-agency-lab-automated-essay-scoring-2/train.csv"
TEST_PATH = "/kaggle/input/learning-agency-lab-automated-essay-scoring-2/test.csv"
OUTPUT_PATH = "/kaggle/working/"

NUM_LABELS = 6
RANDOM_STATE = 42




In [2]:
import os
import numpy as np
import pandas as pd
import warnings

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import cohen_kappa_score, accuracy_score

warnings.filterwarnings("ignore")




In [3]:
# Load data
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

assert {"essay_id", "full_text", "score"}.issubset(train_df.columns)
assert {"essay_id", "full_text"}.issubset(test_df.columns)

# Train/validation split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df["full_text"].astype(str).values,
    train_df["score"].astype(int).values,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=train_df["score"],
)

# TF‑IDF features (more features & 3‑grams)
vectorizer = TfidfVectorizer(
    max_features=100_000,
    ngram_range=(1, 3),
    stop_words="english",
    dtype=np.float32,
)

X_train = vectorizer.fit_transform(train_texts)
X_val = vectorizer.transform(val_texts)

# Ridge regression – use a solver that works with sparse matrices
ridge = Ridge(alpha=0.5, random_state=RANDOM_STATE, solver="lsqr")
ridge.fit(X_train, train_labels)

# Validation predictions
val_pred_cont = ridge.predict(X_val)
val_pred = np.rint(val_pred_cont).astype(int)
val_pred = np.clip(val_pred, 1, NUM_LABELS)

val_qwk = cohen_kappa_score(val_labels, val_pred, weights="quadratic")
val_acc = accuracy_score(val_labels, val_pred)
print(f"Validation QWK: {val_qwk:.5f}")
print(f"Validation Accuracy: {val_acc:.5f}")




Validation QWK: 0.71459
Validation Accuracy: 0.54942


In [4]:
# Retrain on the full training set
full_texts = train_df["full_text"].astype(str).values
full_labels = train_df["score"].astype(int).values

X_full = vectorizer.fit_transform(full_texts)
ridge.fit(X_full, full_labels)




Ridge(alpha=0.5, random_state=42, solver='lsqr')

In [5]:
# Predict on the test set
test_texts = test_df["full_text"].astype(str).values
X_test = vectorizer.transform(test_texts)

test_pred_cont = ridge.predict(X_test)
test_pred = np.rint(test_pred_cont).astype(int)
test_pred = np.clip(test_pred, 1, NUM_LABELS)

submission_df = pd.DataFrame({"essay_id": test_df["essay_id"], "score": test_pred})




In [6]:
# Write submission file
submission_path = os.path.join(OUTPUT_PATH, "submission.csv")
submission_df.to_csv(submission_path, index=False)
print(f"Submission written to {submission_path}")

Submission written to /kaggle/working/submission.csv
